# Nami robot playground

Use the environment containing the **bundled Neuromeka wheel**. Edit the robot IP below.
Motion examples are opt-in. Enabling the gripper initializes it during setup.

Nami is one controller with **22 joint values** and **three task chains**:

| State | Layout |
| --- | --- |
| `q[0:2]` | torso (`Joint_L0`, `Joint_L1`) |
| `q[2:4]` | head |
| `q[4:11]` | left arm |
| `q[11:18]` | right arm |
| `q[18:22]` | four dummy slots |
| `p[0:6]` | head, `arm_index=0` |
| `p[6:12]` | left TCP, `arm_index=1` |
| `p[12:18]` | right TCP, `arm_index=2` |

Joints use degrees; task poses use `[x, y, z, rx, ry, rz]` in mm and xyz Euler degrees.
Servo, stop, recovery, direct teaching, and teleoperation mode affect the whole controller.


In [ ]:
from pathlib import Path
import sys
import time

# Start Jupyter anywhere inside this repository.
deploy_dir = next(
    (candidate for root in (Path.cwd(), *Path.cwd().parents)
     for candidate in (root, root / "deploy")
     if (candidate / "communication" / "robot.py").is_file()), None)
if deploy_dir is None:
    raise RuntimeError("Start Jupyter inside the neuromeka-il repository")
if str(deploy_dir) not in sys.path:
    sys.path.insert(0, str(deploy_dir))

from communication.robot import NamiRobot
from data_collector.etc.nami_teleop import NamiTeleop, NamiTeleopConfig
from helper.extra_utils import ROBOT_STATE


In [ ]:
ROBOT_IP = "192.168.0.180"
ARM_INDICES = (1, 2)  # (1,) for only the left arm; (2,) for only the right.
GRIPPER_ENABLED = False  # Set True to initialize the connected DH gripper.
TOOL_INDEX = 0  # Physical endport; this is not necessarily the arm index.
gripper_config = {
    "enable": GRIPPER_ENABLED,
    "type": "EndportDHGripperClient",
    "params": {
        "robot_ip": ROBOT_IP,
        "tool_index": TOOL_INDEX,
        "speed": 50,
        "force": 50,
    },
}
robot = NamiRobot(robot_ip=ROBOT_IP, gripper_config=gripper_config)


## Read state
The adapter preserves the complete controller state. Use `get_task_pose` to select a TCP.


In [ ]:
state = robot.get_state()
{key: state[key] for key in ("q", "qdot", "p", "pdot", "op_state")}


In [ ]:
{arm: robot.get_task_pose(state, arm) for arm in (0, 1, 2)}


## Controller kinematics
Read-only forward and inverse kinematics queries using the robot controller.


In [ ]:
state = robot.get_state()
robot.compute_forward_kinematics(state["q"], arm_index=1)


In [ ]:
robot.compute_inverse_kinematics(
    robot.get_task_pose(state, 1), state["q"], arm_index=1)


## Optional robot commands
Uncomment individual commands when ready. Refresh the state before changing a target.


In [ ]:
# robot.set_servo_all(enable=True)
# robot.recover()
# robot.set_direct_teaching(enable=True)
# robot.set_direct_teaching(enable=False)

# q = list(robot.get_state()["q"])
# q[4] += 2.0
# robot.move(q, mode="joint_abs", vel_ratio=10, acc_ratio=10, wait=True)

# p = robot.get_task_pose(robot.get_state(), 1)
# p[2] += 2.0
# robot.move(p, mode="task_abs", arm_index=1, vel_ratio=10, acc_ratio=10, wait=True)

# robot.stop_teleop()
# robot.stop_motion(mode="smooth")


## Optional DH gripper control
Set `GRIPPER_ENABLED=True` and select the physical `TOOL_INDEX` above, then rerun
setup to initialize the gripper. Uncomment one command at a time below.
Position values are `1.0` (open), `0.0` (closed), or any value in between.


In [ ]:
# Open:
# robot.move_gripper(mode="no_thread", value=1.0)

# Close:
# robot.move_gripper(mode="no_thread", value=0.0)

# Half-open:
# robot.move_gripper(mode="no_thread", value=0.5)

# Reinitialize after a controller restart:
# robot.gripper_client.initialize()


## Stream a fixed task target
Set `RUN_FIXED_TARGET=True` to stream the current TCP poses with `movetelel_abs`.
The helper starts from idle and stops in `finally`.


In [ ]:
def start_teleop(timeout=10.0):
    robot.stop_teleop()
    deadline = time.monotonic() + timeout
    while robot.get_state()["op_state"] != ROBOT_STATE.IDLE:
        if time.monotonic() >= deadline:
            raise TimeoutError("Controller did not become idle")
        time.sleep(0.05)
    while True:
        robot.start_teleop(mode="task_abs")
        time.sleep(0.2)
        op_state = robot.get_state()["op_state"]
        if op_state == ROBOT_STATE.TELE_OP:
            return
        if ROBOT_STATE.in_failure_state(op_state):
            raise RuntimeError(f"Controller failure: {op_state}")
        if time.monotonic() >= deadline:
            raise TimeoutError("Controller did not enter teleoperation")

RUN_FIXED_TARGET = False
if RUN_FIXED_TARGET:
    session = NamiTeleop(robot, NamiTeleopConfig(arm_indices=ARM_INDICES))
    try:
        start_teleop()
        state = robot.get_state()
        session.reset(state)
        targets = session.device_poses(state)
        for _ in range(20):
            started = time.monotonic()
            command = session.convert(targets, robot.get_state())
            for arm_index, pose in command.items():
                robot.tele_move(pose, mode="task_abs", arm_index=arm_index, vel_scale=0.1, acc_scale=0.5)
            time.sleep(max(0, 0.05 - (time.monotonic() - started)))
    finally:
        robot.stop_teleop()
